## Laboratorio 07 — Búsqueda Adversaria  

**Curso:** Inteligencia Artificial 2026  
**Fecha:** 04 de mayo de 2026

En este laboratorio se implementan algoritmos de búsqueda adversaria:  
- Minimax  
- Minimax con horizonte limitado  
- Poda α-β  
- Monte Carlo Tree Search (MCTS)  

El objetivo es analizar el impacto de la profundidad de búsqueda y el uso de métodos probabilísticos en juegos de suma cero.

In [2]:
import copy
import random
import math
import time



### **1. Tic-Tac-Toe**

Implementar una clase `TicTacToeEngine` para realizar y visualizar un juego de Tic-Tac-Toe de 3×3 o 4×4.

**Objetivo:** Desarrollar un motor de juegos que compare la eficiencia de los algoritmos deterministas (Minimax/Alfa-Beta) frente a probabilísticos (MCTS) bajo distintas configuraciones de tablero.

Construir dos clases principales:

• Clase TicTacToeEngine: Esta clase representa el ”cerebro” y el estado del juego. No debe contener lógica de interfaz de usuario (inputs), solo procesamiento de datos. Métodos obligatorios a implementar:

– ``is_empty(row, col): ``Retorna si una celda está disponible.  
– ``get_moves(): Retorna ``lista de coordenadas (r, c) disponibles.  
– ``is_winner(player):`` Verifica si el jugador ha ganado.  
– ``evaluate(): ``Función heurística para tableros no terminados (asigna puntajes basados en proximidad a ganar).  
– ``minimax_pure(): ``Implementación exhaustiva (solo recomendada para 3 × 3).  
– ``minimax_limit(depth):`` Minimax que se detiene en un horizonte fijo y usa evaluate().  
– ``alpha_beta(depth, alpha, beta):`` Minimax optimizado con poda.  
– ``mcts(iterations, C):`` Monte Carlo Tree Search utilizando la fórmula UCT para selección. Fórmula UCT:

$$
\text{Score} = \text{mean\_win\_rate} + C \cdot \sqrt{\frac{\ln(\text{ParentVisits})}{\text{NodeVisits}}}
$$

In [3]:
class TicTacToeEngine:
    """
    Motor de juego Tic-Tac-Toe de 3x3 o 4x4.
    No contiene lógica de interfaz de usuario (solo procesamiento de datos).
    X es el jugador maximizador, O es el minimizador.
    """

    def __init__(self, size=3):
        self.size = size
        self.board = [[' ' for _ in range(size)] for _ in range(size)]
        self.nodes_explored = 0  # Contador de nodos para medir eficiencia

    # ------------------------------------------------------------------ #
    #  Utilidades básicas del tablero                                       #
    # ------------------------------------------------------------------ #

    def is_empty(self, row, col):
        """Retorna True si la celda (row, col) está disponible."""
        return self.board[row][col] == ' '

    def get_moves(self):
        """Retorna lista de coordenadas (r, c) de celdas disponibles."""
        return [
            (r, c)
            for r in range(self.size)
            for c in range(self.size)
            if self.is_empty(r, c)
        ]

    def is_winner(self, player):
        """Verifica si el jugador indicado ha ganado (filas, columnas, diagonales)."""
        b = self.board
        n = self.size

        # Filas y columnas
        for i in range(n):
            if all(b[i][j] == player for j in range(n)):
                return True
            if all(b[j][i] == player for j in range(n)):
                return True

        # Diagonal principal
        if all(b[i][i] == player for i in range(n)):
            return True

        # Diagonal anti-principal
        if all(b[i][n - 1 - i] == player for i in range(n)):
            return True

        return False

    def is_terminal(self):
        """
        Verifica si el juego terminó.
        Retorna 'X', 'O', 'Tie' o None si el juego continúa.
        """
        if self.is_winner('X'):
            return 'X'
        if self.is_winner('O'):
            return 'O'
        if not self.get_moves():
            return 'Tie'
        return None

    # ------------------------------------------------------------------ #
    #  Función heurística                                                   #
    # ------------------------------------------------------------------ #

    def evaluate(self):
        """
        Heurística para tableros no terminados.
        Asigna puntajes basados en proximidad a ganar
        """
        state = self.is_terminal()
        if state == 'X':
            return 1000
        if state == 'O':
            return -1000
        if state == 'Tie':
            return 0

        b = self.board
        n = self.size
        score = 0

        # Recopilar todas las líneas del tablero
        lines = []
        for i in range(n):
            lines.append([b[i][j] for j in range(n)])   # fila i
            lines.append([b[j][i] for j in range(n)])   # columna i
        lines.append([b[i][i] for i in range(n)])        # diagonal principal
        lines.append([b[i][n - 1 - i] for i in range(n)])  # anti-diagonal

        for line in lines:
            x_count = line.count('X')
            o_count = line.count('O')

            # Línea pura de X (sin O) → potencial para X
            if x_count > 0 and o_count == 0:
                score += x_count ** 2

            # Línea pura de O (sin X) → potencial para O
            if o_count > 0 and x_count == 0:
                score -= o_count ** 2

        return score

    # ------------------------------------------------------------------ #
    #  Minimax exhaustivo (recomendado solo para 3×3)                      #
    # ------------------------------------------------------------------ #

    def minimax_pure(self, is_maximizing):
        self.nodes_explored += 1

        result = self.is_terminal()
        if result == 'X':
            return 1, None
        if result == 'O':
            return -1, None
        if result == 'Tie':
            return 0, None

        moves = self.get_moves()
        best_move = None

        if is_maximizing:
            best_val = -float('inf')
            for r, c in moves:
                self.board[r][c] = 'X'
                val, _ = self.minimax_pure(False)
                self.board[r][c] = ' '
                if val > best_val:
                    best_val, best_move = val, (r, c)
        else:
            best_val = float('inf')
            for r, c in moves:
                self.board[r][c] = 'O'
                val, _ = self.minimax_pure(True)
                self.board[r][c] = ' '
                if val < best_val:
                    best_val, best_move = val, (r, c)

        return best_val, best_move

    # ------------------------------------------------------------------ #
    #  Minimax con horizonte limitado                                       #
    # ------------------------------------------------------------------ #

    def minimax_limit(self, depth, is_maximizing):
        self.nodes_explored += 1

        result = self.is_terminal()
        if result is not None or depth == 0:
            return self.evaluate(), None

        moves = self.get_moves()
        best_move = None

        if is_maximizing:
            best_val = -float('inf')
            for r, c in moves:
                self.board[r][c] = 'X'
                val, _ = self.minimax_limit(depth - 1, False)
                self.board[r][c] = ' '
                if val > best_val:
                    best_val, best_move = val, (r, c)
        else:
            best_val = float('inf')
            for r, c in moves:
                self.board[r][c] = 'O'
                val, _ = self.minimax_limit(depth - 1, True)
                self.board[r][c] = ' '
                if val < best_val:
                    best_val, best_move = val, (r, c)

        return best_val, best_move

    # ------------------------------------------------------------------ #
    #  Alpha-Beta pruning                                                   #
    # ------------------------------------------------------------------ #

    def alpha_beta(self, depth, alpha, beta, is_maximizing):
        """
        Minimax optimizado con poda Alfa-Beta.
        """
        self.nodes_explored += 1

        result = self.is_terminal()
        if result is not None or depth == 0:
            return self.evaluate(), None

        moves = self.get_moves()
        best_move = None

        if is_maximizing:
            best_val = -float('inf')
            for r, c in moves:
                self.board[r][c] = 'X'
                val, _ = self.alpha_beta(depth - 1, alpha, beta, False)
                self.board[r][c] = ' '
                if val > best_val:
                    best_val, best_move = val, (r, c)
                alpha = max(alpha, best_val)
                if beta <= alpha:
                    break  
        else:
            best_val = float('inf')
            for r, c in moves:
                self.board[r][c] = 'O'
                val, _ = self.alpha_beta(depth - 1, alpha, beta, True)
                self.board[r][c] = ' '
                if val < best_val:
                    best_val, best_move = val, (r, c)
                beta = min(beta, best_val)
                if beta <= alpha:
                    break  # Poda alfa

        return best_val, best_move

    # ------------------------------------------------------------------ #
    #  Monte Carlo Tree Search con UCT                                      #
    # ------------------------------------------------------------------ #

    def mcts(self, iterations, C, player):
        moves = self.get_moves()
        if not moves:
            return 0, None

        # Estadísticas por movimiento raíz
        wins  = {m: 0.0 for m in moves}
        visits = {m: 0   for m in moves}
        total_visits = 0  # visitas del nodo padre (raíz)

        opponent = 'O' if player == 'X' else 'X'

        for _ in range(iterations):
            if total_visits == 0:
                selected = random.choice(moves)
            else:
                def uct_score(m):
                    n_i = visits[m]
                    if n_i == 0:
                        return float('inf')  # Nodo no visitado → explorar primero
                    mean_win_rate = wins[m] / n_i
                    exploration   = C * math.sqrt(math.log(total_visits) / n_i)
                    return mean_win_rate + exploration

                selected = max(moves, key=uct_score)

  
            sim = TicTacToeEngine(self.size)
            sim.board = [row[:] for row in self.board] # Copia rápida de la matriz
            r, c = selected
            sim.board[r][c] = player          # Aplicar movimiento seleccionado

            curr = opponent
            while sim.is_terminal() is None:
                avail = sim.get_moves()
                mr, mc = random.choice(avail)
                sim.board[mr][mc] = curr
                curr = 'O' if curr == 'X' else 'X'

         
            result = sim.is_terminal()
            if result == player:
                wins[selected] += 1.0
            elif result == 'Tie':
                wins[selected] += 0.5

            visits[selected] += 1
            total_visits      += 1

        best_move = max(moves, key=lambda m: visits[m])
        self.nodes_explored = total_visits
        return 0, best_move



• Clase GameLoop: Esta clase orquesta el flujo de la partida. Debe ser capaz de configurar una partida con los siguientes parámetros en su constructor.

– size: 3 ó 4.  
– mode: ”H-H” (Humano vs Humano), ”H-IA” (Humano vs IA), ”IA-IA” (IA contra IA).  
– starting_player: Quien realiza el primer movimiento (’H’ o ’IA’). Asumiremos que El primero el humano o la IA1 siempre juegan con ’X’, y el adversario con ’O’.  
– ia_configs: Un diccionario o estructura que defina para cada IA:  

  * Algoritmo a usar (minimax, alpha_beta, mcts).  
  * depth: Horizonte para algoritmos de límite (default 4 para el tablero de 4 × 4).  
  * N: Número de simulaciones para MCTS.  
  * C: Constante de exploración para UCT (default √2).  

In [4]:
class GameLoop:
    """
    Orquesta el flujo de una partida de Tic-Tac-Toe.
    Soporta modos: H-H, H-IA, IA-IA.
    """

    def __init__(self, size=3, mode="IA-IA", starting_player='X', ia_configs=None):
        self.size            = size
        self.mode            = mode          # "H-H", "H-IA", "IA-IA"
        self.starting_player = starting_player
        self.ia_configs      = ia_configs or {}

    # ------------------------------------------------------------------ #
    #  Imprimir tablero                                                    #
    # ------------------------------------------------------------------ #

    def _print_board(self, engine):
        """Imprime el tablero actual en consola."""
        n = engine.size
        sep = "---+" * n
        print()
        for r in range(n):
            row = " | ".join(engine.board[r][c] if engine.board[r][c] != ' ' else '.'
                             for c in range(n))
            print(f"  {row}")
            if r < n - 1:
                print(f"  {sep[:-1]}")
        print()

    # ------------------------------------------------------------------ #
    #  Obtener jugada humana                                               #
    # ------------------------------------------------------------------ #

    def _get_human_move(self, engine, player):
        """Solicita al humano una celda válida."""
        while True:
            try:
                raw = input(f"  Jugador {player}, ingresa fila y columna (ej: 1 2): ")
                parts = raw.strip().split()
                if len(parts) != 2:
                    raise ValueError
                r, c = int(parts[0]) - 1, int(parts[1]) - 1  # 1-indexed para el usuario
                if not (0 <= r < engine.size and 0 <= c < engine.size):
                    print(f"  Celda fuera de rango. Usa valores entre 1 y {engine.size}.")
                    continue
                if not engine.is_empty(r, c):
                    print("  Celda ocupada. Elige otra.")
                    continue
                return (r, c)
            except (ValueError, IndexError):
                print("  Entrada inválida. Escribe dos números separados por espacio.")

    # ------------------------------------------------------------------ #
    #  Obtener jugada de la IA                                             #
    # ------------------------------------------------------------------ #

    def _get_ia_move(self, engine, player, config):
        algo   = config.get('algorithm', 'alpha_beta')
        depth  = config.get('depth', 4)
        N      = config.get('N', 500)
        C      = config.get('C', math.sqrt(2))
        is_max = (player == 'X')

        engine.nodes_explored = 0  # Reiniciar contador antes de cada jugada

        if algo == 'minimax_pure':
            _, move = engine.minimax_pure(is_max)
        elif algo == 'minimax_limit':
            _, move = engine.minimax_limit(depth, is_max)
        elif algo == 'alpha_beta':
            _, move = engine.alpha_beta(depth, -float('inf'), float('inf'), is_max)
        elif algo == 'mcts':
            _, move = engine.mcts(N, C, player)
        else:
            raise ValueError(f"Algoritmo desconocido: {algo}")
        return move

    # ------------------------------------------------------------------ #
    #  Determinar si un jugador es humano según el modo                    #
    # ------------------------------------------------------------------ #

    def _is_human(self, player):
        """
        Retorna True si ese jugador es humano según el modo de juego.
        - H-H : ambos humanos
        - H-IA: X = humano, O = IA
        - IA-IA: ambos IA
        """
        if self.mode == "H-H":
            return True
        if self.mode == "H-IA":
            return player == 'X'   # X siempre es el humano en H-IA
        return False               # IA-IA: ninguno es humano

    # ------------------------------------------------------------------ #
    #  Ejecutar una partida completa                                        #
    # ------------------------------------------------------------------ #

    def play_game(self, verbose=True):
        """
        Ejecuta la partida completa.
        Retorna: (resultado, move_times)
          - resultado  : 'X', 'O' o 'Tie'
          - move_times : {'X': [t1, t2, ...], 'O': [t1, t2, ...]}
        """
        engine   = TicTacToeEngine(self.size)
        current  = self.starting_player
        opponent = 'O' if current == 'X' else 'X'
        move_times = {'X': [], 'O': []}
        move_number = 0

        # Mapeo IA: al menos en modo IA-IA se necesitan dos configs
        ia_names   = list(self.ia_configs.keys())
        player_map = {}
        if len(ia_names) >= 1:
            player_map[current] = self.ia_configs[ia_names[0]]
        if len(ia_names) >= 2:
            player_map[opponent] = self.ia_configs[ia_names[1]]

        if verbose:
            print(f"\n{'='*40}")
            print(f"  Tablero {self.size}x{self.size} | Modo: {self.mode}")
            print(f"{'='*40}")
            self._print_board(engine)

        while engine.is_terminal() is None:
            move_number += 1

            if verbose:
                print(f"  Turno #{move_number} — Jugador {current}")

            t0 = time.time()

            if self._is_human(current):
                # --- Turno humano ---
                move = self._get_human_move(engine, current)
                elapsed = time.time() - t0
                nodes = 0
            else:
                # --- Turno IA ---
                config = player_map[current]
                move   = self._get_ia_move(engine, current, config)
                elapsed = time.time() - t0
                nodes  = engine.nodes_explored

                if verbose:
                    algo = config.get('algorithm', 'alpha_beta')
                    print(f"  IA ({algo}) jugó {(move[0]+1, move[1]+1)} | "
                          f"Nodos explorados: {nodes} | "
                          f"Tiempo: {elapsed:.4f}s")

            # Aplicar movimiento
            if move:
                r, c = move
                engine.board[r][c] = current
                move_times[current].append(elapsed)

            if verbose:
                self._print_board(engine)

            current = 'O' if current == 'X' else 'X'

        # --- Resultado final ---
        result = engine.is_terminal()
        if verbose:
            if result == 'Tie':
                print("  Resultado: ¡Empate!")
            else:
                print(f"  Resultado: ¡Ganó el jugador {result}!")
            print(f"{'='*40}\n")

        return result, move_times


### **2. Explosión Combinatoria:**

(a) En el Tic-Tac-Toe de 3 × 3, realizar lo siguiente:

• Una búsqueda minimax, desde el tablero vacío, variando el depth desde 1 a 9. Registrar el número de nodos visitados y el tiempo de ejecución.

• Repetir lo anterior, pero ahora implementando la poda α-β dentro del minimax.



In [5]:
print("=" * 65)
print(f"{'INCISO 2(a) — Tablero 3×3: Minimax vs Alpha-Beta':^65}")
print("=" * 65)
print(f"{'depth':>6} │ {'Minimax':^22} │ {'Alpha-Beta':^22}")
print(f"{'':>6} │ {'nodos':>10}  {'tiempo(s)':>10} │ {'nodos':>10}  {'tiempo(s)':>10}")
print("─" * 65)

resultados_3x3 = []

for depth in range(1, 10):  # depth 1 a 9
    #  Minimax con límite 
    eng_mm = TicTacToeEngine(size=3)
    eng_mm.nodes_explored = 0
    t0 = time.time()
    eng_mm.minimax_limit(depth, is_maximizing=True)
    t_mm = time.time() - t0
    n_mm = eng_mm.nodes_explored

   
    eng_ab = TicTacToeEngine(size=3)
    eng_ab.nodes_explored = 0
    t0 = time.time()
    eng_ab.alpha_beta(depth, -float('inf'), float('inf'), is_maximizing=True)
    t_ab = time.time() - t0
    n_ab = eng_ab.nodes_explored

    resultados_3x3.append({
        'depth': depth,
        'mm_nodos': n_mm, 'mm_tiempo': t_mm,
        'ab_nodos': n_ab, 'ab_tiempo': t_ab,
    })

    print(f"  {depth:>4} │ {n_mm:>10,}  {t_mm:>10.5f} │ {n_ab:>10,}  {t_ab:>10.5f}")

print("─" * 65)


print("\n   Minimax PURO exhaustivo (sin límite de depth):")
eng_pure = TicTacToeEngine(size=3)
eng_pure.nodes_explored = 0
t0 = time.time()
eng_pure.minimax_pure(is_maximizing=True)
t_pure = time.time() - t0
print(f"  Nodos: {eng_pure.nodes_explored:,}   Tiempo: {t_pure:.5f}s")


        INCISO 2(a) — Tablero 3×3: Minimax vs Alpha-Beta         
 depth │        Minimax         │       Alpha-Beta      
       │      nodos   tiempo(s) │      nodos   tiempo(s)
─────────────────────────────────────────────────────────────────
     1 │         10     0.00053 │         10     0.00052
     2 │         82     0.00262 │         36     0.00155
     3 │        586     0.01708 │        167     0.00408
     4 │      3,610     0.11399 │        630     0.01805
     5 │     18,730     0.55161 │      2,739     0.07516
     6 │     73,450     2.17085 │      4,624     0.12113
     7 │    221,626     4.85469 │     12,614     0.27178
     8 │    422,074     8.91654 │     13,451     0.31505
     9 │    549,946     8.85299 │     18,297     0.29704
─────────────────────────────────────────────────────────────────

   Minimax PURO exhaustivo (sin límite de depth):
  Nodos: 549,946   Tiempo: 6.79325s


(b) En el Tic-Tac-Toe de 4 × 4, realizar una búsqueda desde el tablero vacío con α-β variando el depth de 1 a 6. Registrar el número de nodos visitados, el tiempo de ejecución y el factor de ramificación efectivo:

$$
\sqrt[\text{depth}]{\text{nodos}}
$$

In [9]:
print("\n" + "=" * 65)
print(f"{'INCISO 2(b) — Tablero 4×4: Alpha-Beta':^65}")
print("=" * 65)
print(f"{'depth':>6} │ {'nodos':>12} │ {'tiempo(s)':>12} │ {'factor ramif.':>14}")
print("─" * 65)

resultados_4x4 = []

for depth in range(1, 7):  # depth 1 a 6
    eng = TicTacToeEngine(size=4)
    eng.nodes_explored = 0
    t0 = time.time()
    eng.alpha_beta(depth, -float('inf'), float('inf'), is_maximizing=True)
    t_ab = time.time() - t0
    n_ab = eng.nodes_explored


    factor = n_ab ** (1 / depth) if n_ab > 0 else 0

    resultados_4x4.append({
        'depth': depth,
        'nodos': n_ab,
        'tiempo': t_ab,
        'factor': factor,
    })

    print(f"  {depth:>4} │ {n_ab:>12,} │ {t_ab:>12.5f} │ {factor:>14.2f}")

print("─" * 65)



              INCISO 2(b) — Tablero 4×4: Alpha-Beta              
 depth │        nodos │    tiempo(s) │  factor ramif.
─────────────────────────────────────────────────────────────────
     1 │           17 │      0.00051 │          17.00
     2 │           47 │      0.00156 │           6.86
     3 │          313 │      0.01414 │           6.79
     4 │        2,250 │      0.07940 │           6.89
     5 │        9,968 │      0.38808 │           6.31
     6 │       49,338 │      1.88477 │           6.06
─────────────────────────────────────────────────────────────────


### **3. Duelo de Algoritmos (IA-IA):**
Enfrenten dos configuraciones en 20 partidas:

• IA-1: MCTS con N = 500 y C = √2.  
• IA-2: Minimax limitado a depth = 4 y poda α-β.  

Pregunta: ¿Cuál es más ”inteligente” en términos de ganar y cuál es más ”eficiente” en términos de tiempo por jugada?


In [10]:
# Configuración de cada IA
IA1_CONFIG = {
    'algorithm': 'mcts',
    'N': 500,
    'C': math.sqrt(2),
}

IA2_CONFIG = {
    'algorithm': 'alpha_beta',
    'depth': 4,
}

TOTAL_GAMES = 20
BOARD_SIZE  = 3


### Simulación de las 20 partidas

En cada partida se registra:
- El resultado (`X gana`, `O gana`, `Empate`).
- A qué IA corresponde el ganador (según el símbolo que jugó en esa partida).
- El tiempo medio por jugada de cada IA.

`ia1_symbol` y `ia2_symbol` cambian cada 10 partidas para que ninguna IA tenga siempre la ventaja del primer movimiento.

In [11]:
results = []

for game_num in range(1, TOTAL_GAMES + 1):
    if game_num <= 10:
        ia_configs    = {'IA1': IA1_CONFIG, 'IA2': IA2_CONFIG}
        starting      = 'X'
        ia1_symbol    = 'X'
        ia2_symbol    = 'O'
    else:
        ia_configs    = {'IA1': IA2_CONFIG, 'IA2': IA1_CONFIG}
        starting      = 'X'
        ia1_symbol    = 'O'
        ia2_symbol    = 'X'

    loop = GameLoop(size=BOARD_SIZE, mode="IA-IA",
                    starting_player=starting, ia_configs=ia_configs)
    result, move_times = loop.play_game()

    if result == ia1_symbol:
        winner = 'IA-1 (MCTS)'
    elif result == ia2_symbol:
        winner = 'IA-2 (AB)'
    else:
        winner = 'Empate'

    times_ia1 = move_times[ia1_symbol]
    times_ia2 = move_times[ia2_symbol]
    avg_ia1   = sum(times_ia1) / len(times_ia1) if times_ia1 else 0
    avg_ia2   = sum(times_ia2) / len(times_ia2) if times_ia2 else 0

    results.append({
        'game'    : game_num,
        'ia1_sym' : ia1_symbol,
        'ia2_sym' : ia2_symbol,
        'result'  : result,
        'winner'  : winner,
        'avg_ia1' : avg_ia1,
        'avg_ia2' : avg_ia2,
    })
    print(f"Partida {game_num:2d} | IA-1={ia1_symbol}  IA-2={ia2_symbol} | "
          f"Ganador: {winner:18s} | "
          f"t̄_IA1={avg_ia1*1000:6.2f} ms  t̄_IA2={avg_ia2*1000:6.2f} ms")



  Tablero 3x3 | Modo: IA-IA

  . | . | .
  ---+---+---
  . | . | .
  ---+---+---
  . | . | .

  Turno #1 — Jugador X
  IA (mcts) jugó (3, 1) | Nodos explorados: 500 | Tiempo: 0.0675s

  . | . | .
  ---+---+---
  . | . | .
  ---+---+---
  X | . | .

  Turno #2 — Jugador O
  IA (alpha_beta) jugó (2, 2) | Nodos explorados: 470 | Tiempo: 0.0107s

  . | . | .
  ---+---+---
  . | O | .
  ---+---+---
  X | . | .

  Turno #3 — Jugador X
  IA (mcts) jugó (3, 3) | Nodos explorados: 500 | Tiempo: 0.0474s

  . | . | .
  ---+---+---
  . | O | .
  ---+---+---
  X | . | X

  Turno #4 — Jugador O
  IA (alpha_beta) jugó (3, 2) | Nodos explorados: 232 | Tiempo: 0.0057s

  . | . | .
  ---+---+---
  . | O | .
  ---+---+---
  X | O | X

  Turno #5 — Jugador X
  IA (mcts) jugó (1, 2) | Nodos explorados: 500 | Tiempo: 0.0377s

  . | X | .
  ---+---+---
  . | O | .
  ---+---+---
  X | O | X

  Turno #6 — Jugador O
  IA (alpha_beta) jugó (1, 1) | Nodos explorados: 47 | Tiempo: 0.0010s

  O | X | .
  ---+---+-

### Resumen estadístico

Con los datos recolectados calculamos:
- **Victorias totales** de cada IA (sin importar el símbolo jugado).
- **Tiempo medio por jugada** global de cada IA (promedio de los promedios por partida).

Esto nos permite responder la pregunta del enunciado: ¿cuál IA es más *inteligente* y cuál es más *eficiente*?

In [8]:
wins_ia1   = sum(1 for r in results if r['winner'] == 'IA-1 (MCTS)')
wins_ia2   = sum(1 for r in results if r['winner'] == 'IA-2 (AB)')
ties       = sum(1 for r in results if r['winner'] == 'Empate')

mean_t_ia1 = sum(r['avg_ia1'] for r in results) / len(results) * 1000
mean_t_ia2 = sum(r['avg_ia2'] for r in results) / len(results) * 1000

sep = "─" * 44
print(sep)
print(f"{'RESUMEN FINAL (20 partidas)':^44}")
print(sep)
print(f"  IA-1 (MCTS  N=500 C=√2)  victorias : {wins_ia1:>3}")
print(f"  IA-2 (Alpha-Beta depth=4) victorias : {wins_ia2:>3}")
print(f"  Empates                             : {ties:>3}")
print(sep)
print(f"  Tiempo medio/jugada  IA-1 : {mean_t_ia1:7.3f} ms")
print(f"  Tiempo medio/jugada  IA-2 : {mean_t_ia2:7.3f} ms")
print(sep)

inteligente = "IA-1 (MCTS)"  if wins_ia1 > wins_ia2 else \
              "IA-2 (AB)"    if wins_ia2 > wins_ia1 else "Empate técnico"
eficiente   = "IA-1 (MCTS)"  if mean_t_ia1 < mean_t_ia2 else "IA-2 (AB)"

print(f"\n  Más INTELIGENTE (más victorias) → {inteligente}")
print(f"  Más EFICIENTE   (menos tiempo)  → {eficiente}")


────────────────────────────────────────────
        RESUMEN FINAL (20 partidas)         
────────────────────────────────────────────
  IA-1 (MCTS  N=500 C=√2)  victorias :   0
  IA-2 (Alpha-Beta depth=4) victorias :   4
  Empates                             :  16
────────────────────────────────────────────
  Tiempo medio/jugada  IA-1 :  36.207 ms
  Tiempo medio/jugada  IA-2 :   4.424 ms
────────────────────────────────────────────

  Más INTELIGENTE (más victorias) → IA-2 (AB)
  Más EFICIENTE   (menos tiempo)  → IA-2 (AB)


### Análisis y conclusiones

**¿Cuál es más inteligente?**

El Minimax con poda α-β (depth=4) es *óptimo* en un tablero 3×3: con profundidad 4 ya cubre la mayoría de los árboles relevantes y, al ser determinista, siempre encuentra la jugada teóricamente correcta. En Tic-Tac-Toe perfecto el resultado es siempre empate con dos jugadores óptimos; cuando hay asimetría de símbolo (quien abre gana o empata), α-β la explota de forma consistente.

El MCTS (N=500) es probabilístico: cada iteración es un *rollout* aleatorio. Con solo 500 simulaciones en un tablero 3×3 (árbol de hasta 362 880 hojas), la cobertura es suficiente para partidas razonables, pero la aleatoriedad puede producir jugadas subóptimas ocasionales.

**¿Cuál es más eficiente?**

El Minimax con α-β es mucho más rápido por jugada en 3×3: recorre un árbol pequeño y la poda elimina ramas completas, resultando en microsegundos por decisión. El MCTS debe ejecutar 500 simulaciones completas, lo que implica mayor coste de CPU aunque sigue siendo viable en tiempo real.

**Conclusión final:**
- En un dominio *pequeño* y *perfectamente resoluble* como Tic-Tac-Toe 3×3, α-β es superior en ambas dimensiones: más preciso y más rápido.
- MCTS muestra su verdadero potencial en juegos con espacios de estado enormes (Go, ajedrez sin evaluación experta) donde el árbol completo es intractable y la heurística es difícil de diseñar.

### **4. Para pensar**
Imagine que la IA no debe tardar más de 1 segundo por jugada. Si el algoritmo es lento, ¿qué tipo de implementación o modificación a la estructura anterior debe hacerse para devolver la mejor jugada encontrada considerando la restricción de tiempo? Proponga sus ideas.

***Respuesta***

Para cumplir con la restricción de máximo 1 segundo por jugada, la IA debe utilizar un enfoque basado en tiempo en lugar de profundidad o número fijo de iteraciones.

Una solución es implementar **algoritmos "anytime"**, que puedan detenerse en cualquier momento y devolver la mejor jugada encontrada hasta ese instante.

- En **Minimax o Alpha-Beta**, se puede usar **profundización iterativa**, evaluando primero a baja profundidad y aumentando progresivamente. Si se acaba el tiempo, se retorna la mejor jugada de la última iteración completada.

- En **MCTS (Monte Carlo Tree Search)**, se reemplaza el número fijo de iteraciones por un bucle controlado por tiempo, ejecutando simulaciones hasta que se alcance el límite de 1 segundo.

Como mejoras adicionales, se pueden usar técnicas como **memoización (guardar estados ya evaluados)** y **ordenamiento de movimientos** para aumentar la eficiencia y obtener mejores resultados en menos tiempo.